# Data Preparation

Fase 3 del proceso CRISP-DM. Este notebook transforma los datos crudos en los datasets listos para modelado. Cada decisión técnica está justificada y se resume en la Sección 9, que sirve como referencia directa para la evaluación del proyecto.

## Conexión y carga de datos

In [10]:
import os
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from dotenv import load_dotenv
from IPython.display import display

load_dotenv()

def get_engine():
    user = os.getenv('POSTGRES_USER', 'nba_user')
    pw   = os.getenv('POSTGRES_PASSWORD', 'nba_pass')
    host = os.getenv('POSTGRES_HOST', 'localhost')
    port = os.getenv('POSTGRES_PORT', '5432')
    db   = os.getenv('POSTGRES_DB',   'nba_database')
    return create_engine(f'postgresql+psycopg2://{user}:{pw}@{host}:{port}/{db}')

engine = get_engine()

team_logs   = pd.read_sql('SELECT * FROM fact_team_game_logs',   engine)
player_logs = pd.read_sql('SELECT * FROM fact_player_game_logs', engine)

## Limpieza general

El notebook 02 confirmó que no existen valores nulos ni duplicados en ninguna tabla. Los tipos de datos ya son correctos: `game_date` es `datetime64`, `min` es `float64`, y las columnas enteras están en `int64`. No se requiere ninguna corrección estructural.

In [11]:
pd.DataFrame({
    'filas':   [len(team_logs), len(player_logs)],
    'nulos':   [team_logs.isnull().sum().sum(), player_logs.isnull().sum().sum()],
    'duplicados': [
        team_logs.duplicated(['game_id', 'team_id']).sum(),
        player_logs.duplicated(['game_id', 'player_id']).sum()
    ]
}, index=['team_logs', 'player_logs'])

,filas,nulos,duplicados
team_logs,2460,0,0
player_logs,26651,0,0


## Feature Engineering — Clasificación de equipos

La lógica central de este problema es: para predecir si un equipo ganará el partido del día, el modelo solo puede ver lo que ese equipo hizo en partidos anteriores. Esto impone una regla estricta: los features del partido `t` se calculan exclusivamente con datos de los partidos `1, 2, ..., t-1`.

Se construyen dos escalas temporales —ventanas de 5 y 15 partidos— para capturar tanto el estado de forma inmediato como la tendencia reciente más amplia. Incluir ambas aporta señal complementaria: un equipo puede tener buen promedio en los últimos 15 pero llevar 3 derrotas seguidas, y ambas señales son relevantes.

El primer paso es un auto-join sobre `game_id` para obtener los puntos anotados por el rival en cada partido, que se usarán como feature de defensa del equipo.

In [12]:
opp = team_logs[['game_id', 'team_id', 'pts']].rename(columns={'team_id': 'opp_id', 'pts': 'pts_against'})
tdf = team_logs.merge(opp, on='game_id')
tdf = tdf[tdf['team_id'] != tdf['opp_id']].copy()

tdf['is_home'] = tdf['matchup'].str.contains('vs.', regex=False).astype(int)
tdf['target']  = (tdf['wl'] == 'W').astype(int)
tdf = tdf.sort_values(['team_id', 'game_date']).reset_index(drop=True)

In [13]:
def roll(s, w):
    return s.shift(1).rolling(w, min_periods=w).mean()

roll_cols = ['pts', 'pts_against', 'fg_pct', 'plus_minus', 'reb', 'ast', 'stl']

for w in [5, 10]:
    for col in roll_cols:
        tdf[f'{col}_last{w}'] = tdf.groupby('team_id')[col].transform(roll, w)
    tdf[f'winrate_last{w}'] = tdf.groupby('team_id')['target'].transform(roll, w)

In [14]:
keep_team = (['game_id', 'team_id', 'game_date', 'is_home', 'target'] +
             [c for c in tdf.columns if '_last' in c])
team_feat = tdf[keep_team].dropna().reset_index(drop=True)
team_feat.head(3)

,game_id,team_id,game_date,is_home,target,pts_last5,pts_against_last5,fg_pct_last5,plus_minus_last5,reb_last5,...,stl_last5,winrate_last5,pts_last10,pts_against_last10,fg_pct_last10,plus_minus_last10,reb_last10,ast_last10,stl_last10,winrate_last10
0,0022500206,1610612737,2025-11-10,0,1,116.6,109.6,0.4930,7.0,45.2,...,10.2,0.6,115.2,115.0,0.4754,0.2,42.6,28.3,9.3,0.5
1,0022500223,1610612737,2025-11-12,0,1,112.0,108.4,0.4748,3.6,42.4,...,9.6,0.6,113.9,111.4,0.4784,2.5,43.3,28.8,9.3,0.6
2,0022500227,1610612737,2025-11-13,0,1,116.8,105.0,0.4888,11.8,42.2,...,10.6,0.8,116.1,110.7,0.4878,5.4,43.8,30.4,9.8,0.6


In [15]:
pd.DataFrame({
    'filas originales': [len(tdf)],
    'filas tras dropna': [len(team_feat)],
    'filas eliminadas': [len(tdf) - len(team_feat)],
    'features': [len([c for c in team_feat.columns if '_last' in c or c == 'is_home'])]
})

,filas originales,filas tras dropna,filas eliminadas,features
0,2460,2160,300,17


El dataset de clasificación queda con 2.160 filas. Se pierden 300 filas (10 por equipo × 30 equipos) porque los primeros 10 partidos de cada equipo no tienen historial suficiente para calcular la ventana de 10 partidos. Esta pérdida es intencionada: predecir sin contexto histórico no tiene sentido estadístico.

El resultado son 16 features de rendimiento (8 por cada ventana) más `is_home`, que captura el efecto de localía, uno de los predictores más consistentes en el baloncesto profesional.

## Feature Engineering — Regresión de puntos por jugador

La misma lógica de ventana deslizante se aplica a nivel jugador: los features del partido `t` de un jugador se calculan con sus estadísticas en los partidos `1, ..., t-1`.

Se añade una feature externa: el `defense_rating` del rival, calculado como la media acumulada de puntos permitidos por ese equipo en todos sus partidos anteriores al partido a predecir. Esta variable captura la calidad defensiva del rival y es la única feature que no proviene del historial del jugador sino del contexto del partido.

Antes de cualquier cálculo de rolling, se filtra a partidos con al menos 5 minutos jugados para evitar que apariciones anecdóticas contaminen los promedios móviles. Esta decisión se justifica formalmente en la Sección 5.

In [16]:
player_clean = player_logs[player_logs['min'] >= 5].copy()
player_clean['is_home'] = player_clean['matchup'].str.contains('vs.', regex=False).astype(int)
player_clean = player_clean.sort_values(['player_id', 'game_date']).reset_index(drop=True)

game_opp = tdf[['game_id', 'team_id', 'opp_id']].drop_duplicates()
player_clean = player_clean.merge(game_opp, on=['game_id', 'team_id'], how='left')

defense_df = (tdf[['game_id', 'game_date', 'team_id', 'pts_against']]
              .sort_values(['team_id', 'game_date'])
              .copy())
defense_df['defense_rating'] = defense_df.groupby('team_id')['pts_against'].transform(
    lambda s: s.shift(1).expanding(min_periods=1).mean()
)
defense_map = defense_df[['game_id', 'team_id', 'defense_rating']].rename(columns={'team_id': 'opp_id'})

In [17]:
roll_player = ['pts', 'min', 'fg_pct', 'ft_pct', 'reb', 'ast']

for w in [5, 10]:
    for col in roll_player:
        player_clean[f'{col}_last{w}'] = player_clean.groupby('player_id')[col].transform(roll, w)

player_clean = player_clean.merge(defense_map, on=['game_id', 'opp_id'], how='left')
player_clean['defense_rating'] = player_clean['defense_rating'].fillna(tdf['pts_against'].mean())
player_clean['target'] = player_clean['pts']

keep_player = (['game_id', 'player_id', 'team_id', 'game_date', 'is_home', 'defense_rating', 'target'] +
               [c for c in player_clean.columns if '_last' in c])
player_feat = player_clean[keep_player].dropna().reset_index(drop=True)
player_feat.head(3)

,game_id,player_id,team_id,game_date,is_home,defense_rating,target,pts_last5,min_last5,fg_pct_last5,ft_pct_last5,reb_last5,ast_last5,pts_last10,min_last10,fg_pct_last10,ft_pct_last10,reb_last10,ast_last10
0,0022500395,2544,1610612747,2025-12-20,0,117.037037,36,22.0,35.090000,0.4766,0.5798,7.6,7.8,18.6,33.688833,0.4683,0.5482,5.8,7.5
1,0022500418,2544,1610612747,2025-12-23,0,114.071429,23,27.6,35.419333,0.5368,0.6998,7.2,6.2,21.1,34.488500,0.4648,0.5832,5.9,6.6
2,0022500012,2544,1610612747,2025-12-25,1,112.740741,18,26.4,33.732667,0.4956,0.7088,6.2,6.2,21.7,33.633000,0.4704,0.5877,5.5,6.4


In [18]:
pd.DataFrame({
    'filas originales (min>=5)': [len(player_clean)],
    'filas tras dropna': [len(player_feat)],
    'filas eliminadas': [len(player_clean) - len(player_feat)],
    'features': [len([c for c in player_feat.columns if '_last' in c or c in ['is_home', 'defense_rating']])]
})

,filas originales (min>=5),filas tras dropna,filas eliminadas,features
0,24492,19305,5187,14


El dataset de regresión queda con 19.305 filas. Las pérdidas provienen de dos fuentes: el filtro de minutos (Sección 5) y los primeros 10 partidos de cada jugador sin historial suficiente.

El `defense_rating` se calcula con `expanding().mean()` (media acumulada) en lugar de una media fija de toda la temporada. Esto replica exactamente el conocimiento disponible antes de cada partido: para el tercer juego de la temporada, la defense_rating del rival refleja solo sus dos primeros partidos.

## Tratamiento de outliers

In [19]:
pd.DataFrame({
    'condicion': ['min < 5 (excluidos)', 'min >= 5 (conservados)', 'pts == 0 y min >= 5 (conservados)'],
    'registros': [
        (player_logs['min'] < 5).sum(),
        (player_logs['min'] >= 5).sum(),
        ((player_logs['min'] >= 5) & (player_logs['pts'] == 0)).sum()
    ]
}).assign(
    pct=lambda x: (x['registros'] / len(player_logs) * 100).round(1)
).set_index('condicion')

,registros,pct
condicion,,
min < 5 (excluidos),2159,8.1
min >= 5 (conservados),24492,91.9
pts == 0 y min >= 5 (conservados),1490,5.6


Se eliminan 2.159 registros (8.1% del total) correspondientes a partidos con menos de 5 minutos jugados. Estas apariciones corresponden a jugadores con problemas de faltas, restricciones de entrenador por lesión, o garbage time de último segundo. Incluirlas en las ventanas de predicción distorsionaría el promedio móvil hacia abajo sin aportar señal sobre la capacidad anotadora real del jugador.

Los partidos con 0 puntos y más de 5 minutos se conservan. Representan noches reales de bajo rendimiento —foul trouble, mal tiro, partido difícil— que el modelo debe aprender a anticipar a partir del historial del jugador y el contexto del rival. Eliminarlos introduciría un sesgo al alza en las predicciones.

## Encoding y normalización

La variable `is_home` ya está codificada como entero binario (1 = local, 0 = visitante) desde la etapa de feature engineering, derivada directamente del formato del campo `matchup`. La variable objetivo `target` también está en el formato correcto: 0/1 para clasificación y valor numérico de `pts` para regresión.

In [20]:
pd.DataFrame({
    'target_team (clasificacion)': team_feat['target'].value_counts().to_dict(),
    'is_home_team': team_feat['is_home'].value_counts().to_dict()
})

,target_team (clasificacion),is_home_team
1,1082,1075
0,1078,1085


**Normalización — StandardScaler**

Se elige `StandardScaler` sobre `MinMaxScaler` por las siguientes razones:

1. `MinMaxScaler` requiere conocer los valores mínimo y máximo del rango. Con outliers presentes (partidos con 83 pts o -60 plus_minus), el rango queda distorsionado y la mayoría de los datos se comprimen cerca del extremo inferior.
2. `StandardScaler` escala a media 0 y desviación estándar 1, lo que es más robusto ante outliers extremos y hace que los modelos lineales converjan más rápido.
3. Para modelos basados en árboles (Random Forest, XGBoost), la normalización no afecta al resultado, por lo que usar StandardScaler es neutral para esos modelos.

El scaler **no se aplica en este notebook**. Se integrará como parte del `Pipeline` de scikit-learn en el notebook 04. Esto garantiza que el scaler se ajuste exclusivamente sobre el conjunto de entrenamiento en cada fold de validación cruzada, eliminando cualquier filtración de información del test set.

## División train / test

Con datos temporales, una división aleatoria introduce data leakage sistemático: el modelo podría aprender patrones de partidos de marzo para predecir partidos de octubre, información que no estaba disponible en el momento de la predicción real.

La división es temporal: los datos anteriores al percentil 75 de fechas se usan para entrenamiento, y los posteriores para test. Esto replica la situación real de predicción: el modelo solo ve el pasado para predecir el futuro.

El 25% de test equivale a aproximadamente las últimas 6 semanas de la temporada regular (a partir del 9 de marzo de 2026). Este período no se toca hasta la evaluación final en el notebook 05.

In [21]:
split_date = team_feat['game_date'].sort_values().iloc[int(len(team_feat) * 0.75)]

team_train   = team_feat[team_feat['game_date'] <= split_date].copy()
team_test    = team_feat[team_feat['game_date'] >  split_date].copy()
player_train = player_feat[player_feat['game_date'] <= split_date].copy()
player_test  = player_feat[player_feat['game_date'] >  split_date].copy()

In [22]:
pd.DataFrame({
    'train': [
        len(team_train),
        f"{team_train['target'].mean():.1%} victorias",
        len(player_train),
        f"{player_train['target'].mean():.1f} pts promedio"
    ],
    'test': [
        len(team_test),
        f"{team_test['target'].mean():.1%} victorias",
        len(player_test),
        f"{player_test['target'].mean():.1f} pts promedio"
    ]
}, index=['team filas', 'team target dist', 'player filas', 'player target dist'])

,train,test
team filas,1626,534
team target dist,50.1% victorias,50.0% victorias
player filas,14404,4901
player target dist,11.9 pts promedio,12.0 pts promedio


La fecha de corte es el 9 de marzo de 2026, que corresponde aproximadamente al partido número 62 de cada equipo (75% de 82). Tanto en train como en test, la distribución de victorias se mantiene cerca del 50%, lo que confirma que la división temporal no introduce desbalance artificial en la variable objetivo de clasificación.

## Guardado de datasets procesados

In [23]:
os.makedirs('../data/processed', exist_ok=True)

team_train.to_csv('../data/processed/team_classification_train.csv',   index=False)
team_test.to_csv('../data/processed/team_classification_test.csv',     index=False)
player_train.to_csv('../data/processed/player_regression_train.csv',  index=False)
player_test.to_csv('../data/processed/player_regression_test.csv',    index=False)

pd.DataFrame({
    'archivo': [
        'team_classification_train.csv',
        'team_classification_test.csv',
        'player_regression_train.csv',
        'player_regression_test.csv'
    ],
    'filas': [len(team_train), len(team_test), len(player_train), len(player_test)],
    'columnas': [team_train.shape[1], team_test.shape[1], player_train.shape[1], player_test.shape[1]]
})

,archivo,filas,columnas
0,team_classification_train.csv,1626,21
1,team_classification_test.csv,534,21
2,player_regression_train.csv,14404,19
3,player_regression_test.csv,4901,19


In [24]:
team_train.to_sql('ml_team_classification_train',   engine, if_exists='replace', index=False)
team_test.to_sql('ml_team_classification_test',     engine, if_exists='replace', index=False)
player_train.to_sql('ml_player_regression_train',  engine, if_exists='replace', index=False)
player_test.to_sql('ml_player_regression_test',    engine, if_exists='replace', index=False)

pd.DataFrame({
    'tabla PostgreSQL': [
        'ml_team_classification_train',
        'ml_team_classification_test',
        'ml_player_regression_train',
        'ml_player_regression_test'
    ],
    'estado': ['guardada'] * 4
})

,tabla PostgreSQL,estado
0,ml_team_classification_train,guardada
1,ml_team_classification_test,guardada
2,ml_player_regression_train,guardada
3,ml_player_regression_test,guardada


Los cuatro datasets se guardan tanto en `data/processed/` como en PostgreSQL. Los CSV son la fuente principal para el notebook de modelado y permiten cargar los datos sin depender de la base de datos. Las tablas en PostgreSQL garantizan la trazabilidad completa del pipeline.

## Resumen de decisiones técnicas

| Decisión tomada | Alternativa descartada | Justificación técnica |
|---|---|---|
| División temporal 75/25 | División aleatoria (`train_test_split`) | Los datos tienen dependencia temporal. La división aleatoria introduce data leakage: el modelo podría aprender de partidos de marzo para predecir partidos de octubre, algo imposible en producción real |
| Ventanas de 5 y 10 partidos | Estadísticas acumuladas de toda la temporada | Los promedios de temporada no capturan el estado de forma reciente. Un equipo con buen promedio anual pero en racha de 5 derrotas seguidas requiere ambas escalas para predecirse bien |
| `shift(1)` antes de cada `rolling` | Incluir el partido actual en el cálculo | Sin `shift(1)`, el feature del partido `t` contendría las estadísticas del propio partido `t`. Esto es data leakage directo: se estaría usando la respuesta para construir la pregunta |
| Filtro `min >= 5` antes del rolling | Conservar todos los registros | Apariciones inferiores a 5 minutos no representan el rendimiento real del jugador y sesgan los promedios móviles hacia abajo, introduciendo ruido en lugar de señal |
| `expanding().mean()` para `defense_rating` | Media fija de toda la temporada | La media fija usa datos futuros para caracterizar al rival en partidos tempranos de la temporada. La expanding mean replica el conocimiento real disponible antes de cada partido |
| `StandardScaler` aplicado en Pipeline 04 | `MinMaxScaler` aplicado al dataset | MinMaxScaler es sensible a outliers extremos. StandardScaler es más robusto. Aplicarlo en el Pipeline de sklearn garantiza que se ajusta solo sobre el train set en cada fold de validación cruzada |